### Basic data exploration, cleaning, and resampling of iSUPER 2023, 2024, and 2025 (Jan-Aug) dataset

### EDA on dataset

In [64]:
# import libraries
import pandas as pd
import numpy as np

# list of input files (add more file if required)
files = [
    "iSUPER data/Chelsea DEP MOD-00214 Aug-Nov 2025.csv",
]

# read and stack all the years together
df_list = []

for file in files:
    df_raw = pd.read_csv(file)
    print(f"Loaded {file} with shape {df_raw.shape}")
    # Rename columns -> because we have different flag name spacing in different files
    rename_dict = {}
    for col in df_raw.columns:
        if col.replace(" ", "").lower() == "pm10>150?":
            rename_dict[col] = "pm10 > 150?"
    if rename_dict:
        df_raw = df_raw.rename(columns=rename_dict)

    df_list.append(df_raw)

# concatenate
df_raw = pd.concat(df_list, ignore_index=True)

display(df_raw[2000:3000])
print(df_raw.dtypes)
df_raw.shape

Loaded iSUPER data/Chelsea DEP MOD-00214 Aug-Nov 2025.csv with shape (127325, 16)


,timestamp,timestamp_local,sn,rh,temp,lat,lon,device_state,pm1,pm25,pm10,co,no,no2,o3,pm10 > 150?
2000,2025-08-18T09:20:22Z,2025-08-18T05:20:22Z,MOD-00214,61.6,19.6,42.3874,-71.0252,ACTIVE,3.279,3.750,3.750,167.559,3.089,11.114,16.409,0.0
2001,2025-08-18T09:21:22Z,2025-08-18T05:21:22Z,MOD-00214,61.5,19.6,42.3874,-71.0252,ACTIVE,3.278,4.160,4.816,172.557,3.135,29.986,16.437,0.0
2002,2025-08-18T09:22:22Z,2025-08-18T05:22:22Z,MOD-00214,61.7,19.6,42.3874,-71.0252,ACTIVE,3.230,3.566,3.566,175.288,2.611,26.426,17.363,0.0
2003,2025-08-18T09:23:22Z,2025-08-18T05:23:22Z,MOD-00214,61.8,19.6,42.3874,-71.0252,ACTIVE,3.350,3.698,3.698,175.217,3.124,30.447,16.353,0.0
2004,2025-08-18T09:24:22Z,2025-08-18T05:24:22Z,MOD-00214,61.7,19.6,42.3874,-71.0252,ACTIVE,3.306,3.427,3.427,173.646,3.104,11.047,17.690,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2995,2025-08-19T01:55:21Z,2025-08-18T21:55:21Z,MOD-00214,50.2,18.4,42.3874,-71.0252,ACTIVE,1.735,2.124,12.166,203.095,2.466,31.791,28.174,0.0
2996,2025-08-19T01:56:21Z,2025-08-18T21:56:21Z,MOD-00214,50.6,18.4,42.3874,-71.0252,ACTIVE,1.840,2.311,3.067,185.159,2.551,32.959,27.463,0.0
2997,2025-08-19T01:57:21Z,2025-08-18T21:57:21Z,MOD-00214,51.1,18.4,42.3874,-71.0252,ACTIVE,1.710,2.240,5.782,178.647,2.321,33.175,26.726,0.0
2998,2025-08-19T01:58:21Z,2025-08-18T21:58:21Z,MOD-00214,51.7,18.4,42.3874,-71.0252,ACTIVE,1.719,2.933,2.933,180.684,2.375,34.571,26.276,0.0


timestamp           object
timestamp_local     object
sn                  object
rh                 float64
temp               float64
lat                float64
lon                float64
device_state        object
pm1                float64
pm25               float64
pm10               float64
co                 float64
no                 float64
no2                float64
o3                 float64
pm10 > 150?        float64
dtype: object


(127325, 16)

### Re-sampling and cleaning

We will focus on the some columns in this dataset, hourly sampling is also done (not required for this dataset though but its good to be safe). Missing data is present here, this should be dealt after merging with other datasets.

In [65]:
# rename timestamp_local to timestamp_utc
df_raw["timestamp_utc"] = pd.to_datetime(df_raw["timestamp"], utc=True)

In [66]:
# keep only relevant columns
df_cleaned = df_raw[[
    "timestamp_utc",
    "temp",
    "rh",
    "pm1",
    "pm25",
    "pm10",
    #"pm10 > 150?", # remove due to comment from advisor
]].copy()

display(df_cleaned.head(10))
df_cleaned.shape

,timestamp_utc,temp,rh,pm1,pm25,pm10
0,2025-08-17 00:00:21+00:00,27.1,48.8,15.229,15.316,15.316
1,2025-08-17 00:01:21+00:00,27.1,48.7,14.331,14.804,14.804
2,2025-08-17 00:02:21+00:00,27.1,48.8,13.386,13.386,13.386
3,2025-08-17 00:03:21+00:00,27.1,48.7,14.103,14.103,14.103
4,2025-08-17 00:04:21+00:00,27.1,48.8,14.860,14.938,14.938
5,2025-08-17 00:05:21+00:00,27.0,48.9,14.187,14.264,14.264
6,2025-08-17 00:06:21+00:00,27.0,49.0,13.854,15.130,15.130
7,2025-08-17 00:07:21+00:00,27.0,49.2,13.436,13.521,13.521
8,2025-08-17 00:08:21+00:00,27.0,49.0,14.031,14.117,14.117
9,2025-08-17 00:09:21+00:00,27.0,49.0,14.186,14.186,14.186


(127325, 6)

In [67]:
df_cleaned = df_cleaned.set_index("timestamp_utc").sort_index()
df_cleaned.head(5)

,temp,rh,pm1,pm25,pm10
timestamp_utc,,,,,
2025-08-17 00:00:21+00:00,27.1,48.8,15.229,15.316,15.316
2025-08-17 00:01:21+00:00,27.1,48.7,14.331,14.804,14.804
2025-08-17 00:02:21+00:00,27.1,48.8,13.386,13.386,13.386
2025-08-17 00:03:21+00:00,27.1,48.7,14.103,14.103,14.103
2025-08-17 00:04:21+00:00,27.1,48.8,14.860,14.938,14.938


In [68]:
# In this dataset, out data are already hourly, but we can still resample to enforce a clean hourly format. 
# Keep it consistent with other datasets cleaning script.
agg_rules = {
    "rh": "mean",
    "temp": "mean",
    "pm1": "mean",
    "pm25": "mean",
    "pm10": "mean",
    #"pm10 > 150?": "max", # Here, we use max so that "any exceedance within an hour" would be flag as "1"
}

isuper_5min = df_cleaned.resample("5T").agg(agg_rules)

C:\Users\USER\AppData\Local\Temp\ipykernel_75304\3443841788.py:12: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  isuper_5min = df_cleaned.resample("5T").agg(agg_rules)


In [69]:
isuper_5min = isuper_5min.rename(columns={"temp": "temp_sensor", "rh": "rh_sensor"})

In [70]:
isuper_5min.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 25632 entries, 2025-08-17 00:00:00+00:00 to 2025-11-13 23:55:00+00:00
Freq: 5min
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   rh_sensor    25472 non-null  float64
 1   temp_sensor  25472 non-null  float64
 2   pm1          25472 non-null  float64
 3   pm25         25472 non-null  float64
 4   pm10         25472 non-null  float64
dtypes: float64(5)
memory usage: 1.2 MB


In [71]:
# save csv file as isuper_hourly
isuper_5min.to_csv("Filtered dataset/isuper_5min_full.csv")